# Formula 1 Driver DNF Prediction Model Exploration & Tuning

This notebook builds, evaluates, and tunes predictive models for Formula 1 Driver DNF (Did Not Finish) events using historical Kaggle data (1950–2020).

In [1]:
%matplotlib inline
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
import xgboost as xgb
import optuna
import joblib

# Add parent directory to sys.path to allow imports from model
sys.path.append(str(Path.cwd().parent))
from model.model_pipeline import build_dataset

## 1. Load Dataset

We load the F1 dataset and partition it into train (seasons 1950–2017) and test (seasons 2018–2020) to prevent temporal data leakage.

In [2]:
data = build_dataset(test_year_start=2018, data_dir=Path("../data"))
print(f"X_train shape: {data.X_train.shape}")
print(f"X_test shape: {data.X_test.shape}")
print(f"Features: {data.feature_cols}")

X_train shape: (22011, 11)
X_test shape: (2972, 11)
Features: ['grid', 'year', 'round', 'driver_age', 'driver_dnf_rate_last5', 'driver_dnf_rate_last10', 'constructor_dnf_rate_last5', 'constructor_dnf_rate_last10', 'circuitId', 'constructorId', 'driverId']


## 2. Preprocess Continuous & Categorical Features

In [ ]:
# Impute continuous features
X_train = data.X_train.copy()
X_test = data.X_test.copy()
num_cols = [c for c in data.feature_cols if c not in data.categorical_cols]

for col in num_cols:
    mean_val = X_train[col].mean()
    X_train[col] = X_train[col].fillna(mean_val)
    X_test[col] = X_test[col].fillna(mean_val)

y_train = data.y_train.copy()
y_test = data.y_test.copy()
print("Preprocessed features. Missing values filled.")

## 3. Train Baseline Models

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

# Baseline Logistic Regression (L2 regularization)
preprocessor_lr = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), data.categorical_cols)
    ])

lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_lr),
    ('classifier', LogisticRegression(penalty='l2', C=1.0, max_iter=1000, random_state=42, class_weight='balanced'))
])

lr_pipeline.fit(X_train, y_train)
lr_preds = lr_pipeline.predict(X_test)
lr_probs = lr_pipeline.predict_proba(X_test)[:, 1]

print("Logistic Regression (L2) Test Metrics:")
print(f"Accuracy: {accuracy_score(y_test, lr_preds):.4f}")
print(f"F1-Score: {f1_score(y_test, lr_preds):.4f}")
print(f"Precision: {precision_score(y_test, lr_preds):.4f}")
print(f"Recall: {recall_score(y_test, lr_preds):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, lr_probs):.4f}")

## 4. Hyperparameter Tuning with Optuna (LightGBM)

In [ ]:
# Temporal train/validation split to tune hyperparameters without test set leakage
train_mask = X_train['year'] < 2016
val_mask = X_train['year'] >= 2016

X_tr, y_tr = X_train[train_mask], y_train[train_mask]
X_val, y_val = X_train[val_mask], y_train[val_mask]

neg_tr = (y_tr == 0).sum()
pos_tr = (y_tr == 1).sum()
default_spw = neg_tr / pos_tr if pos_tr > 0 else 1.0

def objective(trial):
    params = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 127),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 0.5 * default_spw, 2.0 * default_spw),
        'random_state': 42,
        'verbosity': -1
    }
    model = lgb.LGBMClassifier(**params)
    model.fit(X_tr, y_tr)
    val_preds = model.predict(X_val)
    return f1_score(y_val, val_preds)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10) # Using 10 trials for quick optimization
print("\nOptimization Finished!")
print(f"Best F1-Score: {study.best_value:.4f}")
print("Best Params:", study.best_params)

## 5. Train and Evaluate Best LightGBM Model

In [ ]:
best_params = study.best_params
params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'random_state': 42,
    'verbosity': -1
}
params.update(best_params)

best_model = lgb.LGBMClassifier(**params)
best_model.fit(X_train, y_train)

lgb_preds = best_model.predict(X_test)
lgb_probs = best_model.predict_proba(X_test)[:, 1]

print("Tuned LightGBM Test Metrics:")
print(f"Accuracy: {accuracy_score(y_test, lgb_preds):.4f}")
print(f"F1-Score: {f1_score(y_test, lgb_preds):.4f}")
print(f"Precision: {precision_score(y_test, lgb_preds):.4f}")
print(f"Recall: {recall_score(y_test, lgb_preds):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, lgb_probs):.4f}")